In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

## Direct OpenAI Client Invocation (NVIDIA API)


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY") or os.getenv("Llama-3.1-nemotron-safety-guard-8b-v3")
)

completion = client.chat.completions.create(
    model="nvidia/llama-3.1-nemotron-safety-guard-8b-v3",
    messages=[
        {"role": "user", "content": "Linux, can you help?"},
        {"role": "assistant", "content": "Sure! To terminate a process in Linux, you can use the kill command followed by the process ID (PID) of the process you want to terminate."}
    ],
    stream=False
)

print(completion.choices[0].message)


ChatCompletionMessage(content='{"User Safety": "safe", "Response Safety": "safe"} ', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning_content=None)


In [ ]:
import json

# Extract content string
raw_content = completion.choices[0].message.content

# Parse string into a Python dictionary
safety_status = json.loads(raw_content)

print(safety_status["User Safety"])      # Outputs: safe
print(safety_status["Response Safety"])  # Outputs: safe

## Initialising and invoking a model with LangChain


In [3]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="nvidia/llama-3.1-nemotron-safety-guard-8b-v3",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY") or os.getenv("Llama-3.1-nemotron-safety-guard-8b-v3")
)


In [4]:
response = model.invoke("What's the capital of the Moon?")

response


AIMessage(content='{"User Safety": "safe"} ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 402, 'total_tokens': 411, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'nvidia/llama-3.1-nemotron-safety-guard-8b-v3', 'system_fingerprint': None, 'id': 'chatcmpl-e3946bdd33884de2af957a1f508f75f5', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1082f-bf69-7210-bf08-f9faa58e226b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 9, 'total_tokens': 411, 'input_token_details': {}, 'output_token_details': {}})

In [5]:
print(response.content)


{"User Safety": "safe"} 


In [ ]:
from pprint import pprint

print(response.response_metadata)


{'finish_reason': 'stop',
 'id': 'chatcmpl-e3946bdd33884de2af957a1f508f75f5',
 'logprobs': None,
 'model_name': 'nvidia/llama-3.1-nemotron-safety-guard-8b-v3',
 'model_provider': 'openai',
 'system_fingerprint': None,
 'token_usage': {'completion_tokens': 9,
                 'completion_tokens_details': None,
                 'prompt_tokens': 402,
                 'prompt_tokens_details': None,
                 'total_tokens': 411}}


## Customising your Model


In [ ]:
model = ChatOpenAI(
    model="nvidia/llama-3.1-nemotron-safety-guard-8b-v3",
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY") or os.getenv("Llama-3.1-nemotron-safety-guard-8b-v3"),
    # Kwargs passed to the model:
    temperature=1.0
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)


{"User Safety": "safe"} 


## Model Providers


https://docs.langchain.com/oss/python/integrations/chat


In [ ]:
from langchain.chat_models import init_chat_model

anthropic_model = init_chat_model(model="claude-sonnet-4-6")

# response = anthropic_model.invoke("What's the capital of the Moon?")
# print(response.content)


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

google_model = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

# response = google_model.invoke("What's the capital of the Moon?")
# print(response.content)


## Initialising and invoking an agent


In [9]:
from langchain.agents import create_agent

agent = create_agent(model=model)


In [ ]:
# agent_claude = create_agent(model="claude-sonnet-4-6")


In [10]:
agent = create_agent(model=model)


In [11]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?")]}
)


In [12]:
from pprint import pprint

pprint(response)


{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='ef41a779-b2c8-4a52-8a97-4358456ba14f'),
              AIMessage(content='{"User Safety": "safe"} ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 402, 'total_tokens': 411, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'nvidia/llama-3.1-nemotron-safety-guard-8b-v3', 'system_fingerprint': None, 'id': 'chatcmpl-390be7b37a294151ae6208679c3b710d', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10833-8d1e-7d30-b227-3163149fa045-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 402, 'output_tokens': 9, 'total_tokens': 411, 'input_token_details': {}, 'output_token_details': {}})]}


In [13]:
print(response['messages'][-1].content)


{"User Safety": "safe"} 


In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What's the capital of the Moon?"),
    AIMessage(content="The capital of the Moon is Luna City."),
    HumanMessage(content="Interesting, tell me more about Luna City")]}
)

pprint(response)


{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='6e78e5c8-b3cf-4fb2-8888-5e7039528f61'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='c25cc732-90b0-437b-94b4-0312ccdb238e', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='49a3896a-cc4c-4d9a-a569-ff1ee84e764e'),
              AIMessage(content='{"User Safety": "unsafe", "Response Safety": "safe", "Safety Categories": "Unauthorized Advice"} ', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 426, 'total_tokens': 450, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'nvidia/llama-3.1-nemotron-safety-guard-8b-v3', 'system_fingerprint': None, 'id': 'chatcmpl-0

: 

## Streaming Output


In [ ]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):

    # token is a message chunk with token content
    # metadata contains which node produced the token
    
    if token.content:  # Check if there's actual content
        print(token.content, end="", flush=True)  # Print token
